In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Silver table
silver_df = spark.table(
    "workspace.movies_dev.silver_movies"
)

# Normalize rating to numeric; non-numeric values (e.g. 'NULL') become null
silver_df = silver_df.withColumn(
    "imdb_rating",
    F.expr("try_cast(imdb_rating as double)")
)

# ------------------------------------------------
# 1. Industry Analytics
# ------------------------------------------------

industry_analytics = (
    silver_df
    .groupBy("industry")
    .agg(
        F.count("*").alias("total_movies"),
        F.round(F.avg("imdb_rating"), 2).alias("average_rating"),
        F.max("imdb_rating").alias("highest_rating"),
        F.min("imdb_rating").alias("lowest_rating")
    )
)

industry_analytics.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.movies_dev.gold_industry_analytics")


# ------------------------------------------------
# 2. Studio Analytics
# Keep different currencies separate
# ------------------------------------------------

studio_analytics = (
    silver_df
    .groupBy("studio", "currency", "unit")
    .agg(
        F.count("*").alias("total_movies"),
        F.sum("budget").alias("total_budget"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("imdb_rating").alias("average_rating")
    )
)

studio_analytics.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.movies_dev.gold_studio_analytics")


# ------------------------------------------------
# 3. Top Rated Movies
# ------------------------------------------------

top_rated_movies = (
    silver_df
    .filter(F.col("imdb_rating").isNotNull())
    .orderBy(
        F.col("imdb_rating").desc(),
        F.col("title").asc()
    )
    .limit(20)
)

top_rated_movies.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.movies_dev.gold_top_rated_movies")


# ------------------------------------------------
# 4. Yearly Analytics
# ------------------------------------------------

yearly_analytics = (
    silver_df
    .groupBy("release_year")
    .agg(
        F.count("*").alias("total_movies"),
        F.round(F.avg("imdb_rating"), 2).alias("average_rating")
    )
    .orderBy("release_year")
)

yearly_analytics.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.movies_dev.gold_yearly_analytics")


# ------------------------------------------------
# Execution summary
# ------------------------------------------------

print("Gold transformations completed successfully")

print("Industry analytics:", industry_analytics.count())
print("Studio analytics:", studio_analytics.count())
print("Top-rated movies:", top_rated_movies.count())
print("Yearly analytics:", yearly_analytics.count())

display(industry_analytics)


In [0]:
%sql
SELECT *
FROM workspace.movies_dev.gold_industry_analytics;

In [0]:
%sql
SELECT *
FROM workspace.movies_dev.gold_studio_analytics;

In [0]:
%sql
SELECT *
FROM workspace.movies_dev.gold_top_rated_movies
ORDER BY imdb_rating DESC;

In [0]:
%sql
SELECT *
FROM workspace.movies_dev.gold_yearly_analytics
ORDER BY release_year;